# Prédire le Compa-Ratio

On charge les salariés, la grille et le marché, et on construit pas à pas un premier modèle qui prédit le Compa-Ratio (salaire / milieu de grille).

---
## 1. Charger les données

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

DOSSIER = '../input'   # le dossier des trois fichiers Excel. '../input/anomalies' : les données avec anomalies. '.' : des fichiers posés à côté de ce notebook.
px.defaults.template = 'plotly_white'   # des graphes sur fond blanc

salaries = pd.read_excel(f'{DOSSIER}/employes.xlsx')
band = pd.read_excel(f'{DOSSIER}/bands.xlsx')
marche = pd.read_excel(f'{DOSSIER}/market.xlsx')

print(f"salaries : {salaries.shape[0]} lignes, {salaries.shape[1]} colonnes")
print(f"band     : {band.shape[0]} lignes, {band.shape[1]} colonnes")
print(f"marche   : {marche.shape[0]} lignes, {marche.shape[1]} colonnes")

---
## 2. Un seul tableau, les ratios et la cible

Chaque salarié reçoit la grille et le marché de son métier et de son niveau. Puis on calcule les ratios.

| Colonne | Calcul | Lecture |
|---|---|---|
| `Compa_Ratio` | salaire / `Mid` | la variable à prédire |
| `Ratio_Grille_Marche` | `Mid` / `Median` | 1,10 = la grille paie 10 % de plus que le marché |
| `Amplitude_Grille` | (`Max` − `Min`) / `Mid` | la largeur de la grille |
| `Age_Embauche` | âge − ancienneté | l'expérience acquise avant d'arriver |

Le tableau garde toutes les colonnes. Le choix de celles qui servent au modèle se fait à l'étape 4, juste avant les corrélations.

In [ ]:
CLES = ['Job_Family', 'Niveau']   # ce qui relie un salarié à sa grille et à son marché
CIBLE = 'Compa_Ratio'             # la variable à prédire

# 1. Un matricule en double, ou une grille en double : on garde la première ligne
salaries = salaries.drop_duplicates('Matricule')
band, marche = band.drop_duplicates(CLES), marche.drop_duplicates(CLES)
for table in (salaries, band, marche):
    # Le métier sans espaces autour, pour qu'il s'écrive pareil dans les trois fichiers
    table['Job_Family'] = table['Job_Family'].astype(str).str.strip()
    # Un chiffre écrit en texte dans Excel (« 12 000 », « n.c. ») devient un nombre, ou une case vide
    for c in ['Fixe_Annuel_MAD', 'Age', 'Anciennete', 'Min', 'Mid', 'Max', 'P25', 'Median', 'P75']:
        if c in table.columns:
            table[c] = pd.to_numeric(table[c], errors='coerce')

# 2. On colle la grille puis le marché à chaque salarié.
#    Une colonne déjà présente chez les salariés (Pays, Devise) n'est pas reprise une deuxième fois.
df = salaries.merge(band[CLES + [c for c in band.columns if c not in salaries.columns]], on=CLES, how='left')
df = df.merge(marche[CLES + [c for c in marche.columns if c not in df.columns]], on=CLES, how='left')

# 3. On calcule les ratios
df['Compa_Ratio'] = df['Fixe_Annuel_MAD'] / df['Mid']
df['Ratio_Grille_Marche'] = df['Mid'] / df['Median']
df['Amplitude_Grille'] = (df['Max'] - df['Min']) / df['Mid']
if 'Age' in df.columns and 'Anciennete' in df.columns:
    df['Age_Embauche'] = df['Age'] - df['Anciennete']
# Une division par zéro donne « infini » : on remet la case à vide
df = df.replace([np.inf, -np.inf], np.nan)
# Un ratio négatif ou nul est impossible (salaire à 0 ou négatif, grille dont le Min dépasse le Max) : case vidée aussi
for c in ['Compa_Ratio', 'Ratio_Grille_Marche', 'Amplitude_Grille']:
    df.loc[df[c] <= 0, c] = np.nan

# 4. Un salarié sans Compa-Ratio (pas de grille, ou pas de salaire) ne peut pas servir : on le retire
sans_cible = df[CIBLE].isna()
df = df[~sans_cible].reset_index(drop=True)
print(f"{len(df)} salariés gardés, {int(sans_cible.sum())} retirés (pas de grille ou pas de salaire).")

# 5. Une colonne vide, ou qui a la même valeur pour tout le monde, ne peut rien expliquer : on la retire.
#    Les nombres à virgule sont arrondis au millième : un écart d'arrondi (0,6600001) n'est pas une vraie différence.
inutiles = [c for c in df.columns if (df[c].round(3) if pd.api.types.is_float_dtype(df[c]) else df[c]).nunique() <= 1]
df = df.drop(columns=inutiles)
if inutiles:
    print(f"Colonnes vides ou identiques pour tous, retirées : {inutiles}")

print(f"\nColonnes du tableau : {list(df.columns)}")
df.head()

---
## 3. Les valeurs aberrantes

Pour chaque colonne de `COLONNES_RETRAIT`, une boîte à moustaches. Les points au-delà des moustaches sont les outliers.

In [ ]:
# Les colonnes où l'on cherche des outliers. Pour en épargner une, effacer son nom ; si on reprend l'âge ou
# l'ancienneté dans le modèle, ajouter 'Age' ou 'Anciennete'.
# Le salaire, la grille et le marché n'y sont pas : un salaire élevé est celui d'un haut niveau, pas une erreur.
COLONNES_RETRAIT = ['Compa_Ratio', 'Age_Embauche', 'Competence_N1']
colonnes_num = [c for c in COLONNES_RETRAIT if c in df.columns]


def outliers(colonne):
    # True pour les valeurs au-delà des moustaches : à plus de 1,5 fois la largeur de la boîte
    q1, q3 = colonne.quantile(0.25), colonne.quantile(0.75)
    return (colonne < q1 - 1.5 * (q3 - q1)) | (colonne > q3 + 1.5 * (q3 - q1))


bilan = pd.DataFrame({'Outliers': [int(outliers(df[c]).sum()) for c in colonnes_num],
                      'Cases vides': [int(df[c].isna().sum()) for c in colonnes_num]}, index=colonnes_num)
bilan['% outliers'] = (bilan['Outliers'] / len(df) * 100).round(1)
display(bilan)

fig = px.box(df[colonnes_num].melt(var_name='Colonne', value_name='Valeur'), x='Valeur', facet_col='Colonne',
             facet_col_wrap=3, height=220 * -(-len(colonnes_num) // 3), title='Une boîte à moustaches par colonne')
fig.update_xaxes(matches=None, showticklabels=True, title_text='')
# Au-dessus de chaque boîte, le nom de la colonne seul (et non « Colonne=Age »)
fig.for_each_annotation(lambda titre: titre.update(text=titre.text.split('=')[-1]))
fig.show()

Traitement : on écarte la ligne. Un salarié qui a un outlier dans une des colonnes de `COLONNES_RETRAIT` sort du tableau, même si ses autres colonnes sont normales.

Avec `ENLEVER = False`, les outliers restent dans le tableau : personne n'est écarté à cette étape.

Les lignes écartées ne sont pas perdues : elles sont rangées dans `ecartes`, avec le matricule et la raison.

In [ ]:
ENLEVER = False # True: une ligne qui a un outlier est écartée. False : les outliers restent dans le tableau.

# 1. Pour chaque ligne, on note pourquoi elle doit partir. Une ligne peut avoir plusieurs raisons.
raison = pd.Series('', index=df.index)
if ENLEVER:   
    for c in [c for c in COLONNES_RETRAIT if c in df.columns]:
        raison[outliers(df[c])] += f'outlier {c} ; '
raison = raison.str.rstrip(' ;')

# 2. On range les lignes à écarter dans « ecartes », avec la raison juste après le matricule
ecartes = df[raison != ''].copy()
ecartes.insert(1, 'Raison', raison[ecartes.index])

# 3. On les retire du tableau. Aucune valeur n'est modifiée.
avant = len(df)
df = df[raison == '']
print(f"{avant} salariés au départ, {len(ecartes)} écartés ({len(ecartes) / avant:.1%}), {len(df)} gardés.")

# 4. Combien de lignes pour chaque raison (une ligne qui a deux raisons est comptée deux fois)
display(ecartes['Raison'].str.split(' ; ').explode().value_counts().to_frame('Lignes écartées'))

La liste des salariés écartés :

In [ ]:
# max_rows=None : toute la liste est affichée, et non seulement le début et la fin
with pd.option_context('display.max_rows', None):
    display(ecartes.sort_values('Raison'))

L'autre façon de faire, laissée en commentaire : au lieu d'écarter la ligne, ramener la valeur à la médiane de sa colonne

In [ ]:
# # 1. Pour chaque colonne à traiter : on remplace ses outliers par sa médiane
# for c in [c for c in COLONNES_RETRAIT if c in df.columns]:
#     mediane, hors = df[c].median(), outliers(df[c])
#     df[c] = df[c].astype(float)
#     df.loc[hors, c] = mediane
#     print(f"{c} : {int(hors.sum())} outliers ramenés à la médiane ({mediane:.2f})")
#
# # 2. Pour toutes les colonnes chiffrées : une case vide prend la médiane de sa colonne
# chiffrees = df.select_dtypes('number').columns
# df[chiffrees] = df[chiffrees].fillna(df[chiffrees].median())

---
## 4. Les corrélations

La liste `COLONNES`, juste en dessous, donne les colonnes utilisées pour prédire le Compa-Ratio. C'est ici qu'on en retire ou qu'on en ajoute :

* retirer une colonne : mettre un `#` au début de sa ligne ;
* reprendre une colonne mise de côté : enlever le `#` ;
* ajouter une colonne du fichier qui n'est pas dans la liste : écrire une nouvelle ligne avec son nom, entre apostrophes et suivi d'une virgule.

Puis relancer cette cellule et toutes celles du dessous : la matrice, le graphe et le modèle se refont avec la nouvelle liste, sans recharger les fichiers.

La performance et le niveau d'études passent en rang (1 = la moins bonne performance, 5 = la meilleure ; 0 = secondaire, 7 = BAC+7), pour apparaître dans la matrice.

In [ ]:
COLONNES = [
    'Job_Family',              # le métier
    'Niveau',                  # le niveau du poste
    'Niveau_Etude_Max',        # le niveau d'études
    'Cat_Ecole_Max',           # la catégorie d'école
    'Age_Embauche',            # âge − ancienneté
    'Competence_N1',           # la note de compétence de l'an dernier
    'Performance_N1',          # la performance de l'an dernier
    'Hot_job',                 # la tension du métier sur le marché
    'Ratio_Grille_Marche',     # Mid / médiane du marché
    'Amplitude_Grille',        # (Max − Min) / Mid

    # ----- Mises de côté pour l'instant -----
    # 'Age',                   # fortement lié à l'ancienneté
    # 'Anciennete',            # remplacée, avec l'âge, par Age_Embauche
    # 'Positionnement_9BOX',   # retiré après le test sur les vraies données
    # 'Sexe',                  # ne doit pas expliquer un salaire
    # 'Entite_N1',             # le pôle
    # 'FTE',                   # le temps de travail
]
# À ne jamais mettre : 'Fixe_Annuel_MAD'. Le Compa-Ratio est calculé avec le salaire : ce serait donner la réponse au modèle.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. On part du tableau complet (df) et on ne garde que la cible et les colonnes de la liste.
#    df lui-même n'est pas modifié : on peut changer la liste et relancer à partir de la cellule du dessus.
absentes = [c for c in COLONNES if c not in df.columns]
if absentes:
    print(f"Colonnes de la liste ignorées (absentes du fichier, ou identiques pour tous les salariés) : {absentes}")
if 'Fixe_Annuel_MAD' in COLONNES:
    print("Fixe_Annuel_MAD ignoré : le Compa-Ratio est calculé avec le salaire, ce serait donner la réponse au modèle.")
donnees = df[[CIBLE] + [c for c in COLONNES if c in df.columns and c not in ('Fixe_Annuel_MAD', CIBLE)]].copy()

# 2. La performance et le niveau d'études passent en rang, pour apparaître dans la matrice.
# Un libellé inconnu compte comme une case vide : il prend le rang médian.
# Si la plupart des libellés du fichier ne sont pas ceux prévus, on ne force rien : la colonne reste
# en texte et passe en 0/1 à l'étape 5.
RANG_PERFORMANCE = {"Très en dessous": 1, "Proche de l'objectif": 2, "A l'objectif": 3, "Supérieur à l'objectif": 4, "Très au-delà": 5}
RANG_ETUDE = {'SECONDAIRE': 0, 'BAC': 1, 'BAC+1': 2, 'BAC+2': 3, 'BAC+3': 4, 'BAC+4': 5, 'BAC+5': 6, 'BAC+6': 6,
              'BAC+7': 7, 'EXPERTISE COMPTABLE': 7}
for colonne, rangs in [('Performance_N1', RANG_PERFORMANCE), ('Niveau_Etude_Max', RANG_ETUDE)]:
    if colonne in donnees.columns and donnees[colonne].isin(list(rangs)).mean() > 0.5:
        donnees[colonne] = donnees[colonne].map(rangs)
        donnees[colonne] = donnees[colonne].fillna(donnees[colonne].median())

# 3. Un salarié qui a une case vide dans une des colonnes chiffrées choisies sort : le modèle ne sait pas lire une case vide
vides = donnees.select_dtypes('number').isna().any(axis=1)
donnees = donnees[~vides]
print(f"{len(donnees)} salariés pour la suite ({int(vides.sum())} retirés pour une case vide dans une des colonnes choisies).")

# 4. La matrice. La cible en premier, puis toutes les colonnes chiffrées
chiffrees = [CIBLE] + [c for c in donnees.select_dtypes('number').columns if c != CIBLE]
corr = donnees[chiffrees].corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, height=700,
                title='Corrélations des colonnes chiffrées')
fig.show()


def liens_et_vif(tableau):
    # Pour chaque colonne utilisée : son lien avec la cible, et son VIF.
    # - Colonne chiffrée : sa corrélation avec la cible, entre −1 et +1 (la première ligne de la matrice).
    # - Colonne en texte (métier, catégorie d'école) : son rapport de corrélation, entre 0 et 1.
    #   0 = toutes les catégories ont le même Compa-Ratio moyen ; 1 = connaître la catégorie suffit à
    #   connaître le Compa-Ratio. Il se compare à une corrélation, sans le signe.
    # Le VIF ne mesure pas l'importance : il dit si la colonne est déjà dite par les AUTRES colonnes chiffrées.
    # 1 = pas du tout ; au-dessus de 5 = à regarder ; au-dessus de 10 = elle répète les autres.
    cible = tableau[CIBLE]
    chiffres = tableau.drop(columns=CIBLE).select_dtypes('number').astype(float)
    avec_constante = np.column_stack([np.ones(len(chiffres)), chiffres.to_numpy()])
    with np.errstate(divide='ignore', invalid='ignore'):
        vif = [variance_inflation_factor(avec_constante, i + 1) for i in range(chiffres.shape[1])]
    resultat = pd.DataFrame({'Type': 'chiffre', 'Lien avec le Compa-Ratio': chiffres.corrwith(cible), 'VIF': vif})
    for c in [c for c in tableau.columns if c != CIBLE and c not in chiffres.columns]:
        # La part des écarts de Compa-Ratio qui tient aux écarts de moyenne entre les catégories, puis sa racine carrée
        moyennes = cible.groupby(tableau[c].fillna('Inconnu')).transform('mean')
        part = ((moyennes - cible.mean()) ** 2).sum() / ((cible - cible.mean()) ** 2).sum()
        resultat.loc[c] = ['catégorie', part ** 0.5, np.nan]
    # Du lien le plus fort au plus faible
    return resultat.sort_values('Lien avec le Compa-Ratio', key=abs, ascending=False)


# Toutes les colonnes utilisées, y compris celles en texte, qui ne peuvent pas entrer dans la matrice
liens = liens_et_vif(donnees)

# La longueur de la barre est la force du lien (sans le signe) ; le chiffre écrit dessus est le lien, avec son signe
barres = liens.assign(Force=liens['Lien avec le Compa-Ratio'].abs()).iloc[::-1]
fig = px.bar(barres, x='Force', y=barres.index, orientation='h', color='Type', text=barres['Lien avec le Compa-Ratio'].round(2),
             color_discrete_map={'chiffre': 'steelblue', 'catégorie': 'darkorange'}, height=40 * len(barres) + 200,
             title='Lien de chaque colonne utilisée avec le Compa-Ratio, du plus fort au plus faible',
             labels={'Force': 'Force du lien (0 = aucun, 1 = total)'})
fig.update_yaxes(categoryorder='array', categoryarray=list(barres.index), title_text='', ticksuffix='  ')
fig.show()

# Le même classement en chiffres, avec le VIF (qui n'existe que pour les colonnes chiffrées)
display(liens.round(2).astype({'VIF': object}).fillna('—'))

Le graphe et le tableau aident à choisir les colonnes :

* **Lien avec le Compa-Ratio** : plus il est loin de 0, plus la colonne compte pour la cible.
* **VIF** : il dit si la colonne répète les autres colonnes explicatives. 1 = non ; au-dessus de 5 = à regarder ; au-dessus de 10 = oui.

**Comprendre le VIF.** Pour chaque colonne, on essaie de la deviner à partir de toutes les autres colonnes explicatives. Le R² de cet essai dit quelle part de la colonne on arrive à deviner, et :

$$\text{VIF} = \frac{1}{1 - R^2}$$

C'est comme une corrélation, mais « 1 contre toutes » au lieu de « 2 à 2 ».

Quand il n'y a que deux colonnes, les deux mesures disent la même chose : le VIF vaut 1 / (1 − r²), avec le r qu'on lit dans la matrice. Une corrélation de 0,9 entre deux colonnes donne un VIF de 5,3.

La différence apparaît dès qu'il y a plus de deux colonnes : le VIF peut être élevé alors qu'aucune case de la matrice ne l'est, parce que la colonne se devine avec une combinaison des autres, et pas avec une seule. Exemple : âge = âge à l'embauche + ancienneté.

On garde une colonne qui a un lien et un VIF bas. Une colonne au VIF élevé et au lien faible peut partir.

Les colonnes en texte (métier, catégorie d'école) ne peuvent pas entrer dans la matrice, mais elles sont dans le graphe et le tableau. Leur lien est un rapport de corrélation, entre 0 et 1 : 0 = toutes les catégories ont le même Compa-Ratio moyen, 1 = connaître la catégorie suffit à connaître le Compa-Ratio. Il se compare à une corrélation, sans le signe. Attention : plus une colonne en texte a de valeurs différentes, plus ce chiffre est gonflé sans raison (avec une valeur par salarié, il vaudrait 1).

Pour changer les colonnes : modifier la liste `COLONNES`, juste au-dessus de la matrice, puis relancer à partir de là.

---
## 5. Feature engineering

Les colonnes en texte qui restent (métier, catégorie d'école) deviennent des colonnes 0/1, une par valeur.

In [ ]:
# 1. Les colonnes en texte deviennent des colonnes 0/1. Une case vide devient la valeur « Inconnu ».
#    Pour une colonne à 3 valeurs, 2 colonnes 0/1 suffisent : la valeur la plus fréquente n'a pas de colonne,
#    elle sert de référence et les autres se lisent par rapport à elle. Pourquoi la plus fréquente : avec une
#    petite référence, les colonnes 0/1 se répètent entre elles et leur VIF gonfle sans raison.
en_texte = [c for c in donnees.columns if not pd.api.types.is_numeric_dtype(donnees[c])]
# Une colonne en texte qui a trop de valeurs différentes (un intitulé de poste, par exemple) créerait autant
# de colonnes 0/1, presque toutes à 0 : elle n'est pas utilisée.
MAX_VALEURS = 30
trop_de_valeurs = [c for c in en_texte if donnees[c].nunique() > MAX_VALEURS]
if trop_de_valeurs:
    print(f"Colonnes en texte non utilisées (plus de {MAX_VALEURS} valeurs différentes) : {trop_de_valeurs}")
    donnees = donnees.drop(columns=trop_de_valeurs)
    en_texte = [c for c in en_texte if c not in trop_de_valeurs]
print(f"Colonnes en texte passées en 0/1 : {en_texte}")
donnees[en_texte] = donnees[en_texte].fillna('Inconnu').astype(str)
references = [f'{c}_{donnees[c].mode()[0]}' for c in en_texte]
print(f"Valeurs de référence (sans colonne 0/1) : {references}")
donnees = pd.get_dummies(donnees, columns=en_texte, dtype=float).drop(columns=references)

# 2. Les features qui nous suivent jusqu'au bout, et la cible
X, y = donnees.drop(columns=CIBLE), donnees[CIBLE]
print(f"\n{X.shape[1]} features pour {X.shape[0]} salariés :")
print(list(X.columns))

---
## 6. Le modèle

80 % des salariés servent à apprendre (train), 20 % à vérifier (test). Le modèle est une régression linéaire.

In [ ]:
# 1. On coupe en train et test. random_state : le même découpage à chaque lancement.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train : {len(X_train)} salariés. Test : {len(X_test)} salariés.")

# 2. Le modèle apprend sur le train
modele = LinearRegression().fit(X_train, y_train)

# 3. Il prédit le Compa-Ratio des salariés du test, qu'il n'a jamais vus
y_pred = modele.predict(X_test)

# 4. Les erreurs : le vrai Compa-Ratio moins le Compa-Ratio prédit
erreurs = y_test - y_pred
resultats = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred, 'erreur': erreurs})
display(resultats.head(10).round(3))

# 5. Les métriques, calculées sur le test
metriques = pd.Series({'MAE (erreur moyenne)': mean_absolute_error(y_test, y_pred),
                       'MSE': mean_squared_error(y_test, y_pred),
                       'RMSE': mean_squared_error(y_test, y_pred) ** 0.5,
                       'R²': r2_score(y_test, y_pred)})
display(metriques.round(4).to_frame('Sur le test'))

* **MAE** : l'erreur moyenne, en points de Compa-Ratio. 0,05 = le modèle se trompe de 5 % du milieu de grille en moyenne.
* **RMSE** : pareil, mais les grosses erreurs comptent plus.
* **R²** : la part des écarts de Compa-Ratio que le modèle explique. 1 = tout, 0 = pas mieux que donner la moyenne à tout le monde.

In [ ]:
# Le Compa-Ratio prédit face au vrai : un bon modèle met les points le long de la diagonale
fig = px.scatter(resultats, x='y_test', y='y_pred', opacity=0.4, height=650, title='Compa-Ratio prédit face au vrai (test)',
                 labels={'y_test': 'Vrai Compa-Ratio (y_test)', 'y_pred': 'Compa-Ratio prédit (y_pred)'})
bornes = [resultats['y_test'].min(), resultats['y_test'].max()]
fig.add_scatter(x=bornes, y=bornes, mode='lines', name='Prédiction parfaite', line=dict(color='black'))
fig.show()

# Les erreurs : un bon modèle les serre autour de 0
fig = px.histogram(resultats, x='erreur', nbins=60, height=450, title='Répartition des erreurs (test)',
                   labels={'erreur': 'Erreur (y_test − y_pred)'})
fig.add_vline(x=0, line_color='black')
fig.show()